# SATF backbone + MCD (Maximum Classifier Discrepancy) UDA: test→four

本版保留现有 SATF 特征提取器，只替换跨域训练机制为 MCD。

MCD 核心三步：
```text
A. G + C1 + C2：在 source 上最小化 CE
B. 固定 G：C1/C2 保持 source 分类，同时最大化 target 两分类器分歧
C. 固定 C1/C2：只更新 G，最小化 target 两分类器分歧；重复 K 次
```

严格协议：
- `test_2~test_7`：source EEG + label
- `four_1~four_4`：target EEG only
- `test_8` / `four_5`：全部训练完成后才读取
- 不用 FBCCA / CCA / 人工频率模板 / target label / pseudo-label

说明：MCD 三步机制与 discrepancy 定义按官方实现；SATF backbone、EEG 数据加载和学习率是本任务移植，不声称复现原论文整网。


In [1]:
from pathlib import Path
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import resample
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cpu


## 1. Strict cross-domain protocol: test=source, four=target


In [2]:
DATA_DIR = Path("data")

# Source session: labeled
TRAIN_FILES = [f"test_{i}_ssvep" for i in range(2, 8)]
TEST_FILES = ["test_8_ssvep"]

# Target session: unlabeled adaptation + locked final test
ADAPTATION_FILES = [f"four_{i}_ssvep" for i in range(1, 5)]
DRIFT_FILES = ["four_5_ssvep"]

FREQUENCIES_HZ = np.array([15.0, 12.0, 10.0])
NUM_CLASSES = len(FREQUENCIES_HZ)

SAMPLE_RATE_HZ = 250.0
FOUR_DELAY_SECONDS = 0.5
FOUR_WINDOW_SECONDS = 1.5
MODEL_SAMPLES = 372
EEG_CHANNELS = 8
VALIDATION_FRACTION = 0.20

BATCH_SIZE = 64
SOURCE_BATCH_SIZE = BATCH_SIZE // 2
TARGET_BATCH_SIZE = BATCH_SIZE // 2
MAX_EPOCHS = 500
LEARNING_RATE = 1e-3
LR_PATIENCE = 20
EARLY_STOP_PATIENCE = 60


# MCD settings
STAGE1_MAX_EPOCHS = 250
STAGE1_LEARNING_RATE = 1e-3

STAGE2_EPOCHS = 60
MCD_FEATURE_LR = 1e-4
MCD_CLASSIFIER_LR = 1e-4
MCD_DISCREPANCY_WEIGHT = 1.0
MCD_NUM_GENERATOR_STEPS = 4   # official MCD default num_k=4


TOPK_RATIOS = (0.50, 2 / 3, 0.75)
KAN_GRID_SIZE = 5
KAN_SPLINE_ORDER = 3

roles = TRAIN_FILES + TEST_FILES + ADAPTATION_FILES + DRIFT_FILES
assert len(roles) == len(set(roles)), "file leakage across roles"
assert set(TRAIN_FILES).isdisjoint(TEST_FILES + ADAPTATION_FILES + DRIFT_FILES)
assert set(TEST_FILES).isdisjoint(ADAPTATION_FILES + DRIFT_FILES)
assert set(ADAPTATION_FILES).isdisjoint(DRIFT_FILES)

MODEL_FROZEN = False

print("TWO-SESSION UDA FILE ROLES")
print("  source labeled train/internal val:", TRAIN_FILES)
print("  source diagnostic locked until freeze:", TEST_FILES)
print("  target unlabeled adaptation:", ADAPTATION_FILES)
print("  target final test locked until freeze:", DRIFT_FILES)
print("  source domain=test, target domain=four")

TWO-SESSION UDA FILE ROLES
  source labeled train/internal val: ['test_2_ssvep', 'test_3_ssvep', 'test_4_ssvep', 'test_5_ssvep', 'test_6_ssvep', 'test_7_ssvep']
  source diagnostic locked until freeze: ['test_8_ssvep']
  target unlabeled adaptation: ['four_1_ssvep', 'four_2_ssvep', 'four_3_ssvep', 'four_4_ssvep']
  target final test locked until freeze: ['four_5_ssvep']
  source domain=test, target domain=four


In [3]:
def prepare_trials(raw, name):
    x = np.stack(raw) if np.asarray(raw).dtype == object else np.asarray(raw)
    if x.ndim != 3 or x.shape[1] != EEG_CHANNELS:
        raise ValueError(f"{name}: expected [trial,{EEG_CHANNELS},time], got {x.shape}")

    if name.startswith("four_"):
        start = round(FOUR_DELAY_SECONDS * SAMPLE_RATE_HZ)
        requested = round(FOUR_WINDOW_SECONDS * SAMPLE_RATE_HZ)
        if x.shape[-1] <= start:
            raise ValueError(f"{name}: cannot skip {start} samples")
        x = x[:, :, start:min(start + requested, x.shape[-1])]
    elif not name.startswith("test_"):
        raise ValueError(f"{name}: unknown session prefix")

    original_samples = x.shape[-1]
    if original_samples != MODEL_SAMPLES:
        x = resample(x, MODEL_SAMPLES, axis=-1)

    x = np.asarray(x, dtype=np.float32)
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    x = (x - mean) / np.maximum(std, 1e-6)
    return x, original_samples


def load_labeled_files(file_names):
    # Before freeze, only source TRAIN_FILES may be read with labels.
    if not MODEL_FROZEN:
        if not set(file_names).issubset(set(TRAIN_FILES)):
            raise RuntimeError(
                "Before freeze, labeled loader may only read source TRAIN_FILES."
            )
    else:
        allowed = set(TEST_FILES + DRIFT_FILES)
        if not set(file_names).issubset(allowed):
            raise RuntimeError("After freeze, final labeled loader only accepts TEST/DRIFT files.")

    records = []
    for name in file_names:
        x, original_samples = prepare_trials(
            np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True), name
        )
        y = np.load(DATA_DIR / f"eeg_labels_{name}.npy").astype(np.int64)

        if len(x) != len(y):
            raise ValueError(f"{name}: X/y mismatch")
        if not np.isin(y, np.arange(NUM_CLASSES)).all():
            raise ValueError(f"{name}: labels must be 0..{NUM_CLASSES-1}")

        records.append((x, y, name))
        print("labeled", name, "original", original_samples,
              "shape", x.shape, "counts", np.bincount(y, minlength=NUM_CLASSES))
    return records


def load_target_unlabeled_files(file_names):
    # Critical UDA rule: this function has no label path at all.
    if MODEL_FROZEN:
        raise RuntimeError("Target adaptation loader is training-only.")
    if not set(file_names).issubset(set(ADAPTATION_FILES)):
        raise RuntimeError("Unlabeled target loader may only read ADAPTATION_FILES.")

    records = []
    for name in file_names:
        x, original_samples = prepare_trials(
            np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True), name
        )
        records.append((x, name))
        print("UNLABELED target", name, "original", original_samples, "shape", x.shape)
    return records


def concatenate_labeled(records):
    return (
        np.concatenate([x for x, _, _ in records]),
        np.concatenate([y for _, y, _ in records]),
        np.concatenate([np.full(len(y), name, dtype=object) for _, y, name in records]),
    )


def concatenate_unlabeled(records):
    return (
        np.concatenate([x for x, _ in records]),
        np.concatenate([np.full(len(x), name, dtype=object) for x, name in records]),
    )

In [4]:
# Source labeled data
source_records = load_labeled_files(TRAIN_FILES)
X_pool, y_pool, source_pool = concatenate_labeled(source_records)

idx = np.arange(len(y_pool))
strata = np.array([f"{s}::{y}" for s, y in zip(source_pool, y_pool)])

train_idx, val_idx = train_test_split(
    idx,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=strata,
)
assert set(train_idx).isdisjoint(val_idx)

X_train, y_train = X_pool[train_idx], y_pool[train_idx]
X_validation, y_validation = X_pool[val_idx], y_pool[val_idx]

# Target session UDA: EEG ONLY, no labels loaded.
target_records = load_target_unlabeled_files(ADAPTATION_FILES)
X_target_adapt, target_source = concatenate_unlabeled(target_records)

print("source train / validation:", len(y_train), len(y_validation))
print("target unlabeled adaptation:", len(X_target_adapt))
print("test_8/four_5 remain LOCKED")

labeled test_2_ssvep original 374 shape (19, 8, 372) counts [8 4 7]
labeled test_3_ssvep original 373 shape (41, 8, 372) counts [11 16 14]
labeled test_4_ssvep original 374 shape (69, 8, 372) counts [17 23 29]
labeled test_5_ssvep original 373 shape (102, 8, 372) counts [30 35 37]
labeled test_6_ssvep original 373 shape (106, 8, 372) counts [17 43 46]
labeled test_7_ssvep original 373 shape (67, 8, 372) counts [19 20 28]
UNLABELED target four_1_ssvep original 373 shape (46, 8, 372)
UNLABELED target four_2_ssvep original 372 shape (152, 8, 372)
UNLABELED target four_3_ssvep original 372 shape (95, 8, 372)
UNLABELED target four_4_ssvep original 372 shape (122, 8, 372)
source train / validation: 323 81
target unlabeled adaptation: 415
test_8/four_5 remain LOCKED


## 2. Paper sinusoidal position encoding

In [5]:
class PaperSinusoidalPositionEncoding(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        pe = torch.zeros(1, 1, channels, samples, dtype=torch.float32)
        position = torch.arange(samples, dtype=torch.float32)

        pair_index = torch.arange((channels + 1)//2, dtype=torch.float32)
        denominator = torch.pow(10000.0, (2.0 * pair_index) / channels)

        for c in range(channels):
            i = c // 2
            phase = position / denominator[i]
            pe[0, 0, c] = torch.sin(phase) if c % 2 == 0 else torch.cos(phase)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe

## 3. PESA: 1×1 Conv + 3×3 DW-Conv, 3 heads, channel Top-k

In [6]:
class QKVProjection(nn.Module):
    def __init__(self, heads):
        super().__init__()
        self.pointwise = nn.Conv2d(1, heads, kernel_size=1, bias=False)
        self.depthwise = nn.Conv2d(
            heads, heads, kernel_size=3, padding=1, groups=heads, bias=False
        )

    def forward(self, x):
        return self.depthwise(self.pointwise(x))


class PaperPESA(nn.Module):
    def __init__(self, channels, heads=3, ratios=TOPK_RATIOS):
        super().__init__()
        self.channels = channels
        self.heads = heads

        # PAPER_UNSPECIFIED_ASSUMPTION: floor rounding
        self.k_values = tuple(
            max(1, min(channels, math.floor(channels * ratio)))
            for ratio in ratios
        )

        self.q_proj = QKVProjection(heads)
        self.k_proj = QKVProjection(heads)
        self.v_proj = QKVProjection(heads)

        # PAPER_UNSPECIFIED_ASSUMPTION:
        # paper says adaptive fusion but gives no exact implementation.
        self.scale_logits = nn.Parameter(torch.zeros(len(self.k_values)))

        self.out_proj = nn.Conv2d(heads, 1, kernel_size=1, bias=False)

    @staticmethod
    def sparse_attention(scores, values, topk):
        indices = scores.topk(topk, dim=-1).indices
        mask = torch.zeros_like(scores, dtype=torch.bool)
        mask.scatter_(-1, indices, True)
        sparse_scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(sparse_scores, dim=-1)
        return weights @ values, indices

    def forward(self, x, return_indices=False):
        # [B,H,C,T]
        q = F.normalize(self.q_proj(x), p=2, dim=-1)
        k = F.normalize(self.k_proj(x), p=2, dim=-1)
        v = self.v_proj(x)

        # Paper formula uses d=C/Nh.
        d = self.channels / self.heads
        scores = (q @ k.transpose(-1, -2)) / math.sqrt(d)  # [B,H,C,C]

        branches, selections = [], []
        for topk in self.k_values:
            branch, indices = self.sparse_attention(scores, v, topk)
            branches.append(branch)
            selections.append(indices)

        alpha = torch.softmax(self.scale_logits, dim=0)
        fused = sum(w * b for w, b in zip(alpha, branches))

        # Figure 1 shows a residual path.
        out = x + self.out_proj(fused)

        if return_indices:
            return out, selections, scores
        return out

## 4. TCN branch: exact Table-1 shapes

In [7]:
class BSiLU(nn.Module):
    def __init__(self, alpha=1.67):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return (x + self.alpha) * torch.sigmoid(x) - self.alpha / 2.0


class PaperTCN(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(channels,1), bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # PAPER_INTERNAL_INCONSISTENCY:
        # text says p=23 -> PyTorch output 16;
        # table requires output 15, so use p=22.
        self.conv2 = nn.Conv2d(
            16, 16, kernel_size=(1,samples), stride=(1,3), padding=(0,22), bias=False
        )
        self.bn2 = nn.BatchNorm2d(16)

        self.conv3 = nn.Conv2d(16, 16, kernel_size=(1,12), bias=False)
        self.bn3 = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=(1,4), bias=False)
        self.bn4 = nn.BatchNorm2d(32)

        self.act = BSiLU()

        # Paper figure/text: 32-D Conv4 feature -> FC/MLP -> 4-D branch feature.
        self.feature_fc = nn.Linear(32, 4)

    def forward(self, x, return_shapes=False):
        shapes = {}

        x = self.act(self.bn1(self.conv1(x)))
        shapes["conv1"] = tuple(x.shape)

        x = self.act(self.bn2(self.conv2(x)))
        shapes["conv2"] = tuple(x.shape)

        x = self.act(self.bn3(self.conv3(x)))
        shapes["conv3"] = tuple(x.shape)

        x = self.act(self.bn4(self.conv4(x)))
        shapes["conv4"] = tuple(x.shape)

        feature = self.feature_fc(x.flatten(1))
        shapes["feature"] = tuple(feature.shape)

        return (feature, shapes) if return_shapes else feature

## 5. FCN branch: Spatial Conv → 1×1 → FFT → KAN

In [8]:
class SplineKANLinear(nn.Module):
    # PAPER_UNSPECIFIED_ASSUMPTION:
    # paper does not provide KAN grid/order/layer/library.
    def __init__(self, in_features, out_features,
                 grid_size=KAN_GRID_SIZE, spline_order=KAN_SPLINE_ORDER,
                 grid_min=-3.0, grid_max=3.0):
        super().__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order
        self.grid_min = grid_min
        self.grid_max = grid_max

        step = (grid_max - grid_min) / grid_size
        knots = (
            torch.arange(-spline_order, grid_size + spline_order + 1,
                         dtype=torch.float32) * step + grid_min
        )
        self.register_buffer("knots", knots)

        self.base = nn.Linear(in_features, out_features)
        basis_count = grid_size + spline_order
        self.spline_weight = nn.Parameter(
            torch.empty(out_features, in_features, basis_count)
        )
        nn.init.normal_(self.spline_weight, mean=0.0, std=0.02)

    def basis(self, x):
        x = self.grid_max * torch.tanh(x / self.grid_max)
        x = x.unsqueeze(-1)
        k = self.knots

        basis = ((x >= k[:-1]) & (x < k[1:])).to(x.dtype)
        eps = torch.finfo(x.dtype).eps

        for degree in range(1, self.spline_order + 1):
            left_den = (k[degree:-1] - k[:-(degree+1)]).clamp_min(eps)
            right_den = (k[degree+1:] - k[1:-degree]).clamp_min(eps)

            left = ((x - k[:-(degree+1)]) / left_den) * basis[..., :-1]
            right = ((k[degree+1:] - x) / right_den) * basis[..., 1:]
            basis = left + right

        return basis

    def forward(self, x):
        base = self.base(F.silu(x))
        spline = torch.einsum("bin,oin->bo", self.basis(x), self.spline_weight)
        return base + spline


class PaperFCN(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        self.spatial_conv = nn.Conv2d(
            1, 32, kernel_size=(channels,1), bias=False
        )
        self.pointwise = nn.Conv2d(
            32, 16, kernel_size=1, bias=False
        )

        fft_bins = samples // 2 + 1
        self.kan = SplineKANLinear(16 * fft_bins, 4)

    def forward(self, x, return_shapes=False):
        shapes = {}

        x = self.spatial_conv(x)
        shapes["spatial_conv"] = tuple(x.shape)

        x = self.pointwise(x)
        shapes["pointwise"] = tuple(x.shape)

        spectrum = torch.abs(torch.fft.rfft(x, dim=-1))
        shapes["fft"] = tuple(spectrum.shape)

        feature = self.kan(spectrum.flatten(1))
        shapes["feature"] = tuple(feature.shape)

        return (feature, shapes) if return_shapes else feature

## 6. SATF shared feature generator + two independent classifiers


In [9]:

class SATFMCD(nn.Module):
    def __init__(self):
        super().__init__()
        self.pe = PaperSinusoidalPositionEncoding(EEG_CHANNELS, MODEL_SAMPLES)
        self.pesa = PaperPESA(EEG_CHANNELS, heads=3)
        self.tcn = PaperTCN(EEG_CHANNELS, MODEL_SAMPLES)
        self.fcn = PaperFCN(EEG_CHANNELS, MODEL_SAMPLES)

        # Shared feature generator G -> 16-D
        self.fusion_fc = nn.Linear(8, 16)
        self.fusion_bn = nn.BatchNorm1d(16)
        self.fusion_act = BSiLU()

        # Two independently initialized task classifiers C1/C2
        self.classifier1 = nn.Linear(16, NUM_CLASSES)
        self.classifier2 = nn.Linear(16, NUM_CLASSES)

    def extract_features(self, eeg, return_debug=False):
        if eeg.ndim == 3:
            eeg = eeg.unsqueeze(1)
        if eeg.ndim != 4:
            raise ValueError(f"expected [B,C,T] or [B,1,C,T], got {eeg.shape}")

        x = self.pe(eeg)

        if return_debug:
            x, selections, scores = self.pesa(x, return_indices=True)
            tfeat, tshapes = self.tcn(x, return_shapes=True)
            ffeat, fshapes = self.fcn(x, return_shapes=True)
        else:
            x = self.pesa(x)
            tfeat = self.tcn(x)
            ffeat = self.fcn(x)

        fused = torch.cat([tfeat, ffeat], dim=1)
        embedding = self.fusion_act(self.fusion_bn(self.fusion_fc(fused)))

        if return_debug:
            return embedding, {
                "pesa_output_shape": tuple(x.shape),
                "attention_shape": tuple(scores.shape),
                "topk": self.pesa.k_values,
                "selections": selections,
                "tcn": tshapes,
                "fcn": fshapes,
                "embedding_shape": tuple(embedding.shape),
            }
        return embedding

    def classify_embedding(self, z):
        return self.classifier1(z), self.classifier2(z)

    def forward(self, eeg, return_embedding=False, return_debug=False):
        if return_debug:
            z, debug = self.extract_features(eeg, return_debug=True)
            l1, l2 = self.classify_embedding(z)
            debug["logits1"] = l1
            debug["logits2"] = l2
            return debug

        z = self.extract_features(eeg)
        l1, l2 = self.classify_embedding(z)

        if return_embedding:
            return l1, l2, z
        return l1, l2


def mcd_discrepancy(logits1, logits2):
    """
    Official MCD-style discrepancy:
        mean(abs(softmax(C1) - softmax(C2)))
    """
    p1 = F.softmax(logits1, dim=1)
    p2 = F.softmax(logits2, dim=1)
    return torch.mean(torch.abs(p1 - p2))


model = SATFMCD().to(DEVICE)

# Parameter groups are explicit so Stage B/C can update exactly the intended part.
generator_params = []
classifier_params = []
for name, p in model.named_parameters():
    if name.startswith("classifier1.") or name.startswith("classifier2."):
        classifier_params.append(p)
    else:
        generator_params.append(p)

print(model)
print("PESA top-k:", model.pesa.k_values)
print("generator parameters:", sum(p.numel() for p in generator_params))
print("classifier parameters:", sum(p.numel() for p in classifier_params))
print("total trainable parameters:", sum(p.numel() for p in model.parameters()))


SATFMCD(
  (pe): PaperSinusoidalPositionEncoding()
  (pesa): PaperPESA(
    (q_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (k_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (v_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (out_proj): Conv2d(3, 1, kernel_size=(1, 1), stride=(1, 1), bias=False)
  )
  (tcn): PaperTCN(
    (conv1): Conv2d(1, 16, kernel_size=(8, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=Tr

## 7. Structural check only — no synthetic accuracy


In [10]:

model.eval()
with torch.no_grad():
    structural_input = torch.zeros(4, EEG_CHANNELS, MODEL_SAMPLES, device=DEVICE)
    dbg = model(structural_input, return_debug=True)

print("input:", tuple(structural_input.shape))
print("embedding:", dbg["embedding_shape"])
print("classifier1:", tuple(dbg["logits1"].shape))
print("classifier2:", tuple(dbg["logits2"].shape))

assert dbg["embedding_shape"] == (4, 16)
assert dbg["logits1"].shape == (4, NUM_CLASSES)
assert dbg["logits2"].shape == (4, NUM_CLASSES)

d0 = mcd_discrepancy(dbg["logits1"], dbg["logits1"])
assert torch.allclose(d0, torch.zeros_like(d0), atol=1e-8)

print("MCD structural assertions: PASS")


input: (4, 8, 372)
embedding: (4, 16)
classifier1: (4, 3)
classifier2: (4, 3)
MCD structural assertions: PASS


## 8. Stage 1 source pretraining → Stage 2 MCD adaptation


In [11]:

source_train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    ),
    batch_size=SOURCE_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

target_train_loader = DataLoader(
    TensorDataset(torch.tensor(X_target_adapt, dtype=torch.float32)),
    batch_size=TARGET_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED + 1),
)


def infinite_batches(loader):
    while True:
        for batch in loader:
            yield batch


@torch.no_grad()
def predict_probs(eeg):
    model.eval()
    x = torch.tensor(eeg, dtype=torch.float32, device=DEVICE)
    l1, l2 = model(x)
    p1 = F.softmax(l1, dim=1)
    p2 = F.softmax(l2, dim=1)
    pe = 0.5 * (p1 + p2)
    return (
        p1.cpu().numpy(),
        p2.cpu().numpy(),
        pe.cpu().numpy(),
    )


def source_val_ensemble_accuracy():
    _, _, pe = predict_probs(X_validation)
    pred = pe.argmax(axis=1)
    return float(np.mean(pred == y_validation))


# =========================================================
# Stage 1: source-only supervised pretraining
# Train G, C1, C2 together from independent random heads.
# =========================================================
print("\n===== STAGE 1: MCD SOURCE SUPERVISED PRETRAINING =====")

for p in model.parameters():
    p.requires_grad_(True)

stage1_optimizer = torch.optim.Adam(
    list(generator_params) + list(classifier_params),
    lr=STAGE1_LEARNING_RATE,
)

stage1_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    stage1_optimizer,
    mode="max",
    factor=0.5,
    patience=LR_PATIENCE,
)

stage1_best_val = -1.0
stage1_best_state = None
stage1_no_improve = 0

for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    total = 0
    ens_correct = 0

    for source_eeg, source_labels in source_train_loader:
        source_eeg = source_eeg.to(DEVICE)
        source_labels = source_labels.to(DEVICE)

        l1, l2 = model(source_eeg)
        loss1 = F.cross_entropy(l1, source_labels)
        loss2 = F.cross_entropy(l2, source_labels)
        loss = loss1 + loss2

        stage1_optimizer.zero_grad()
        loss.backward()
        stage1_optimizer.step()

        with torch.no_grad():
            p = 0.5 * (
                F.softmax(l1, dim=1) +
                F.softmax(l2, dim=1)
            )
            ens_correct += (p.argmax(1) == source_labels).sum().item()

        loss_sum += loss.item() * len(source_labels)
        total += len(source_labels)

    val_acc = source_val_ensemble_accuracy()
    stage1_scheduler.step(val_acc)

    if val_acc > stage1_best_val:
        stage1_best_val = val_acc
        stage1_best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        stage1_no_improve = 0
    else:
        stage1_no_improve += 1

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"stage1 epoch={epoch:03d} "
            f"loss={loss_sum/max(total,1):.4f} "
            f"src_train_ensemble={ens_correct/max(total,1):.4f} "
            f"src_val_ensemble={val_acc:.4f} "
            f"lr={stage1_optimizer.param_groups[0]['lr']:.2e}"
        )

    if stage1_no_improve >= EARLY_STOP_PATIENCE:
        print("stage1 early stop at epoch", epoch)
        break

assert stage1_best_state is not None
model.load_state_dict(stage1_best_state)
model.to(DEVICE)
model.eval()

print("stage1 best source validation ensemble:", stage1_best_val)

stage1_frozen_state = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}


# =========================================================
# Stage 2: Maximum Classifier Discrepancy
# =========================================================
print("\n===== STAGE 2: MCD UNSUPERVISED ADAPTATION =====")

opt_g = torch.optim.Adam(generator_params, lr=MCD_FEATURE_LR)
opt_c = torch.optim.Adam(classifier_params, lr=MCD_CLASSIFIER_LR)

steps_per_epoch = max(len(source_train_loader), len(target_train_loader))

for epoch in range(1, STAGE2_EPOCHS + 1):
    source_iter = infinite_batches(source_train_loader)
    target_iter = infinite_batches(target_train_loader)

    sum_a = 0.0
    sum_b = 0.0
    sum_c = 0.0
    src_correct = 0
    src_total = 0

    for _ in range(steps_per_epoch):
        source_eeg, source_labels = next(source_iter)
        (target_eeg,) = next(target_iter)

        source_eeg = source_eeg.to(DEVICE)
        source_labels = source_labels.to(DEVICE)
        target_eeg = target_eeg.to(DEVICE)

        # -------------------------------------------------
        # Step A: update G, C1, C2 to classify SOURCE
        # -------------------------------------------------
        for p in generator_params + classifier_params:
            p.requires_grad_(True)

        model.train()
        opt_g.zero_grad()
        opt_c.zero_grad()

        s1, s2 = model(source_eeg)
        loss_a = (
            F.cross_entropy(s1, source_labels) +
            F.cross_entropy(s2, source_labels)
        )
        loss_a.backward()
        opt_g.step()
        opt_c.step()

        with torch.no_grad():
            sp = 0.5 * (
                F.softmax(s1, dim=1) +
                F.softmax(s2, dim=1)
            )
            src_correct += (sp.argmax(1) == source_labels).sum().item()
            src_total += len(source_labels)

        # -------------------------------------------------
        # Step B: freeze G; update C1/C2
        # Keep source CE small while MAXIMIZING target discrepancy.
        # -------------------------------------------------
        for p in generator_params:
            p.requires_grad_(False)
        for p in classifier_params:
            p.requires_grad_(True)

        opt_c.zero_grad()

        # Generator is frozen in this step.
        with torch.no_grad():
            zs = model.extract_features(source_eeg)
            zt = model.extract_features(target_eeg)

        s1_b, s2_b = model.classify_embedding(zs)
        t1_b, t2_b = model.classify_embedding(zt)

        source_ce_b = (
            F.cross_entropy(s1_b, source_labels) +
            F.cross_entropy(s2_b, source_labels)
        )
        dis_b = mcd_discrepancy(t1_b, t2_b)

        loss_b = source_ce_b - MCD_DISCREPANCY_WEIGHT * dis_b
        loss_b.backward()
        opt_c.step()

        # -------------------------------------------------
        # Step C: freeze C1/C2; update G to MINIMIZE
        # target classifier discrepancy. Repeat K times.
        # -------------------------------------------------
        for p in generator_params:
            p.requires_grad_(True)
        for p in classifier_params:
            p.requires_grad_(False)

        last_dis_c = None
        for _k in range(MCD_NUM_GENERATOR_STEPS):
            opt_g.zero_grad()

            zt = model.extract_features(target_eeg)
            t1_c, t2_c = model.classify_embedding(zt)
            dis_c = mcd_discrepancy(t1_c, t2_c)

            dis_c.backward()
            opt_g.step()
            last_dis_c = dis_c.detach()

        # Restore for next batch.
        for p in classifier_params:
            p.requires_grad_(True)

        sum_a += float(loss_a.detach())
        sum_b += float(loss_b.detach())
        sum_c += float(last_dis_c)

    src_val = source_val_ensemble_accuracy()

    if epoch == 1 or epoch % 10 == 0 or epoch == STAGE2_EPOCHS:
        print(
            f"stage2 epoch={epoch:03d} "
            f"stepA_srcCE={sum_a/steps_per_epoch:.5f} "
            f"stepB_objective={sum_b/steps_per_epoch:.5f} "
            f"stepC_target_dis={sum_c/steps_per_epoch:.6f} "
            f"src_train_ensemble={src_correct/max(src_total,1):.4f} "
            f"src_val_ensemble_monitor={src_val:.4f}"
        )

# Fixed final epoch: no target-label checkpoint selection.
stage2_frozen_state = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

print("Stage 2 completed with fixed", STAGE2_EPOCHS, "epochs")
print("MCD generator steps K:", MCD_NUM_GENERATOR_STEPS)
print("four_1~four_4 used as UNLABELED target EEG only")
print("test_8/four_5 still LOCKED")



===== STAGE 1: MCD SOURCE SUPERVISED PRETRAINING =====
stage1 epoch=001 loss=2.2513 src_train_ensemble=0.3839 src_val_ensemble=0.4321 lr=1.00e-03
stage1 epoch=020 loss=0.5043 src_train_ensemble=0.9783 src_val_ensemble=0.9012 lr=1.00e-03
stage1 epoch=040 loss=0.0712 src_train_ensemble=1.0000 src_val_ensemble=0.9506 lr=5.00e-04
stage1 epoch=060 loss=0.0738 src_train_ensemble=1.0000 src_val_ensemble=0.9259 lr=2.50e-04
stage1 early stop at epoch 67
stage1 best source validation ensemble: 0.9876543209876543

===== STAGE 2: MCD UNSUPERVISED ADAPTATION =====
stage2 epoch=001 stepA_srcCE=0.54661 stepB_objective=0.45143 stepC_target_dis=0.081016 src_train_ensemble=1.0000 src_val_ensemble_monitor=1.0000
stage2 epoch=010 stepA_srcCE=0.39435 stepB_objective=0.35026 stepC_target_dis=0.018389 src_train_ensemble=0.9974 src_val_ensemble_monitor=0.9753
stage2 epoch=020 stepA_srcCE=0.28917 stepB_objective=0.27546 stepC_target_dis=0.010631 src_train_ensemble=1.0000 src_val_ensemble_monitor=0.9630
stage2

## 9. Freeze, then unlock test_8 / four_5 exactly once


In [12]:

for p in model.parameters():
    p.requires_grad_(False)
model.eval()
MODEL_FROZEN = True

print("all model parameters frozen; held-out files unlocked")

source_test_records = load_labeled_files(TEST_FILES)
drift_test_records = load_labeled_files(DRIFT_FILES)

X_source_test, y_source_test, _ = concatenate_labeled(source_test_records)
X_drift_test, y_drift_test, _ = concatenate_labeled(drift_test_records)


all model parameters frozen; held-out files unlocked
labeled test_8_ssvep original 373 shape (67, 8, 372) counts [23 24 20]
labeled four_5_ssvep original 372 shape (125, 8, 372) counts [25 44 56]


In [13]:

def report(name, labels, pred):
    correct = int(np.sum(pred == labels))
    acc = correct / len(labels)
    bal = balanced_accuracy_score(labels, pred)
    recalls = recall_score(
        labels,
        pred,
        labels=np.arange(NUM_CLASSES),
        average=None,
        zero_division=0,
    )
    cm = confusion_matrix(
        labels,
        pred,
        labels=np.arange(NUM_CLASSES),
    )

    print(f"\n{name}")
    print(f"accuracy: {acc:.4f} ({correct}/{len(labels)})")
    print(f"balanced accuracy: {bal:.4f}")
    for f, r in zip(FREQUENCIES_HZ, recalls):
        print(f"recall {f:g} Hz: {r:.4f}")
    print("confusion matrix rows=true [15,12,10], cols=pred [15,12,10]")
    print(cm)
    return acc, bal, recalls, cm


@torch.no_grad()
def predict_state_all(state, eeg):
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()

    x = torch.tensor(eeg, dtype=torch.float32, device=DEVICE)
    l1, l2 = model(x)
    p1 = F.softmax(l1, dim=1)
    p2 = F.softmax(l2, dim=1)
    pe = 0.5 * (p1 + p2)

    return {
        "C1": p1.argmax(1).cpu().numpy(),
        "C2": p2.argmax(1).cpu().numpy(),
        "Ensemble": pe.argmax(1).cpu().numpy(),
    }


results = {}

for stage_name, state in [
    ("Stage1 source-only", stage1_frozen_state),
    ("Stage2 MCD", stage2_frozen_state),
]:
    print("\n================", stage_name, "================")

    src_preds = predict_state_all(state, X_source_test)
    tgt_preds = predict_state_all(state, X_drift_test)

    results[stage_name] = {"source": {}, "target": {}}

    for head in ["C1", "C2", "Ensemble"]:
        results[stage_name]["source"][head] = report(
            f"{stage_name} | {head} | test_8 source diagnostic",
            y_source_test,
            src_preds[head],
        )
        results[stage_name]["target"][head] = report(
            f"{stage_name} | {head} | four_5 final target",
            y_drift_test,
            tgt_preds[head],
        )

print("\nENSEMBLE TARGET DELTA (Stage2 - Stage1)")
base = results["Stage1 source-only"]["target"]["Ensemble"]
adapt = results["Stage2 MCD"]["target"]["Ensemble"]

print("accuracy delta:", adapt[0] - base[0])
print("balanced-accuracy delta:", adapt[1] - base[1])

# Restore Stage2 state for optional PESA inspection.
model.load_state_dict(stage2_frozen_state)
model.to(DEVICE)
model.eval()



================ Stage1 source-only ================

Stage1 source-only | C1 | test_8 source diagnostic
accuracy: 0.8955 (60/67)
balanced accuracy: 0.8982
recall 15 Hz: 0.8696
recall 12 Hz: 0.8750
recall 10 Hz: 0.9500
confusion matrix rows=true [15,12,10], cols=pred [15,12,10]
[[20  1  2]
 [ 1 21  2]
 [ 1  0 19]]

Stage1 source-only | C1 | four_5 final target
accuracy: 0.4000 (50/125)
balanced accuracy: 0.3695
recall 15 Hz: 0.2400
recall 12 Hz: 0.3864
recall 10 Hz: 0.4821
confusion matrix rows=true [15,12,10], cols=pred [15,12,10]
[[ 6  8 11]
 [12 17 15]
 [14 15 27]]

Stage1 source-only | C2 | test_8 source diagnostic
accuracy: 0.8060 (54/67)
balanced accuracy: 0.8143
recall 15 Hz: 0.8261
recall 12 Hz: 0.6667
recall 10 Hz: 0.9500
confusion matrix rows=true [15,12,10], cols=pred [15,12,10]
[[19  0  4]
 [ 6 16  2]
 [ 1  0 19]]

Stage1 source-only | C2 | four_5 final target
accuracy: 0.3840 (48/125)
balanced accuracy: 0.3585
recall 15 Hz: 0.2800
recall 12 Hz: 0.2955
recall 10 Hz: 0.5000

SATFMCD(
  (pe): PaperSinusoidalPositionEncoding()
  (pesa): PaperPESA(
    (q_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (k_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (v_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (out_proj): Conv2d(3, 1, kernel_size=(1, 1), stride=(1, 1), bias=False)
  )
  (tcn): PaperTCN(
    (conv1): Conv2d(1, 16, kernel_size=(8, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=Tr

## 10. PESA selections on final trials

In [14]:
sample = torch.tensor(X_drift_test[:3], dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    dbg = model(sample, return_debug=True)

for topk, indices in zip(dbg["topk"], dbg["selections"]):
    indices = indices.cpu().numpy()
    print(f"top-{topk} shape={indices.shape}")
    for trial_idx in range(len(sample)):
        print(f"  trial {trial_idx+1}: {indices[trial_idx].tolist()}")

top-4 shape=(3, 3, 8, 4)
  trial 1: [[[1, 3, 5, 6], [2, 3, 6, 7], [3, 5, 7, 6], [2, 3, 7, 6], [3, 7, 5, 4], [7, 4, 5, 6], [5, 7, 6, 2], [7, 4, 6, 2]], [[2, 5, 4, 1], [3, 2, 6, 7], [4, 0, 7, 6], [0, 1, 7, 5], [6, 0, 2, 1], [0, 3, 1, 2], [0, 4, 7, 1], [0, 3, 1, 6]], [[0, 5, 3, 4], [0, 1, 5, 2], [0, 2, 3, 5], [3, 0, 4, 6], [4, 5, 0, 2], [5, 0, 4, 2], [6, 4, 7, 2], [4, 7, 6, 0]]]
  trial 2: [[[1, 6, 4, 2], [2, 6, 4, 7], [3, 6, 7, 0], [2, 7, 6, 3], [3, 7, 5, 6], [7, 4, 5, 6], [5, 7, 6, 3], [7, 4, 6, 0]], [[2, 1, 6, 7], [3, 2, 0, 6], [4, 0, 6, 7], [0, 5, 7, 1], [6, 0, 2, 5], [0, 3, 2, 4], [0, 4, 7, 2], [0, 6, 3, 2]], [[0, 4, 6, 3], [0, 1, 2, 5], [0, 2, 3, 5], [3, 0, 6, 4], [4, 0, 5, 7], [5, 0, 3, 4], [6, 4, 7, 0], [7, 4, 6, 3]]]
  trial 3: [[[1, 5, 3, 7], [2, 5, 3, 7], [3, 5, 7, 1], [2, 7, 3, 5], [3, 7, 5, 4], [7, 4, 5, 6], [5, 7, 6, 2], [7, 4, 6, 2]], [[2, 1, 4, 6], [3, 7, 6, 2], [4, 0, 6, 3], [0, 7, 5, 1], [6, 0, 2, 5], [0, 3, 1, 2], [0, 4, 7, 1], [0, 1, 3, 6]], [[0, 5, 2, 3], [0, 1, 2, 5]

## Reporting boundary

这版必须称为：**SATF backbone + MCD UDA (test→four)**。

已发表/官方实现支持的部分：
- 双分类器 C1/C2；
- Step A source supervised；
- Step B classifier 最大化 target discrepancy，同时维持 source 分类；
- Step C generator 最小化 target discrepancy；
- discrepancy = mean(abs(softmax(C1)-softmax(C2)))；
- Step C 默认重复 K=4。

本任务移植部分：SATF backbone、EEG preprocessing、Adam/LR、60 epochs。

最终固定报告 C1/C2/Ensemble 全部结果；不能根据 `four_5` 结果事后选择其中一个。
